# 🚀 HUẤN LUYỆN YOLOV11 100 EPOCHS TRÊN GOOGLE COLAB (GPU T4)
Dự án phân loại 7 lớp rác: `battery`, `cardboard`, `paper`, `glass`, `metal`, `plastic`, `organic`.
Tự động giải nén `trash (3).zip` từ Google Drive, huấn luyện 100 Epochs với Data Augmentation chống Overfitting và lưu `best.pt` về Drive.

### 📌 BƯỚC 1: Kiểm tra GPU & Cài đặt thư viện Ultralytics YOLOv11

In [ ]:
# 1. Kiểm tra card GPU T4 của Google Colab
!nvidia-smi

# 2. Cài đặt thư viện Ultralytics mới nhất
!pip install -q ultralytics

### 📌 BƯỚC 2: Kết nối Google Drive để đọc `trash (3).zip`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 📌 BƯỚC 3: Tự động tìm và giải nén `trash (3).zip` vào ổ cứng Colab

In [ ]:
import os
import glob
import zipfile

# Tìm file zip trên Google Drive
possible_zip_names = [
    '/content/drive/MyDrive/trash (3).zip',
    '/content/drive/MyDrive/trash(3).zip',
    '/content/drive/MyDrive/trash (2).zip',
    '/content/drive/MyDrive/trash (1).zip',
    '/content/drive/MyDrive/trash.zip',
    '/content/drive/MyDrive/Trash_dataset_balanced.zip'
]

zip_file = None
for p in possible_zip_names:
    if os.path.exists(p):
        zip_file = p
        break

if not zip_file:
    # Tìm kiếm tự động bất kỳ file zip nào có chữ trash trong MyDrive
    found = glob.glob('/content/drive/MyDrive/*trash*.zip') + glob.glob('/content/drive/MyDrive/*Trash*.zip')
    if found:
        zip_file = found[0]

if zip_file and os.path.exists(zip_file):
    print(f'⚡ Đã tìm thấy: {zip_file}')
    print('📦 Đang giải nén siêu tốc vào /content/trash_project...')
    with zipfile.ZipFile(zip_file, 'r') as z:
        z.extractall('/content/trash_project')
    print('✅ Giải nén hoàn tất!')
else:
    print('❌ Không tìm thấy file zip! Các file zip hiện có trong Drive của bạn:')
    print(glob.glob('/content/drive/MyDrive/*.zip'))

### 📌 BƯỚC 4: Tự động định vị thư mục Dataset & Tạo `data.yaml` chuẩn

In [ ]:
# Tìm thư mục chứa train/images
dataset_root = None
for root, dirs, files in os.walk('/content/trash_project'):
    if 'train' in dirs and os.path.exists(os.path.join(root, 'train', 'images')):
        dataset_root = root
        break

if not dataset_root:
    dataset_root = '/content/trash_project'

print(f'📍 Thư mục Dataset gốc: {dataset_root}')

# Tạo file data.yaml trên Colab trỏ đúng thư mục
yaml_content = f"""
path: {dataset_root}
train: train/images
val: val/images
test: test/images

nc: 7
names:
  0: battery
  1: cardboard
  2: paper
  3: glass
  4: metal
  5: plastic
  6: organic
"""

with open('/content/data.yaml', 'w') as f:
    f.write(yaml_content.strip())

print('✅ Đã tạo /content/data.yaml thành công!')

### 📌 BƯỚC 5: Huấn luyện YOLO11s 100 Epochs + Data Augmentation Chống Overfitting

In [ ]:
from ultralytics import YOLO

# Load pre-trained base weights YOLO11s
model = YOLO('yolo11s.pt')

# Bắt đầu Huấn luyện 100 Epochs
results = model.train(
    data='/content/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    patience=30,
    # --- CHỐNG OVERFITTING & TĂNG CƯỜNG DỮ LIỆU ---
    weight_decay=0.0005,
    dropout=0.1,
    label_smoothing=0.1,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.3,
    scale=0.5,
    degrees=15.0,
    fliplr=0.5,
    close_mosaic=15,
    # --- LƯU TRỰC TIẾP VÀO GOOGLE DRIVE ---
    project='/content/drive/MyDrive/Trash_YOLO11_Project',
    name='trash_yolo11s_100epochs',
    save=True,
    plots=True,
    val=True
)

print('🎉 HUẤN LUYỆN 100 EPOCHS HOÀN TẤT!')
print('Trọng số tốt nhất đã được lưu tại: /content/drive/MyDrive/Trash_YOLO11_Project/trash_yolo11s_100epochs/weights/best.pt')

### 📌 BƯỚC 6: Đánh giá Model trên tập Test

In [ ]:
# Đánh giá mAP trên tập test
best_path = '/content/drive/MyDrive/Trash_YOLO11_Project/trash_yolo11s_100epochs/weights/best.pt'
eval_model = YOLO(best_path)
metrics = eval_model.val(data='/content/data.yaml', split='test')

print(f'🏆 mAP@50    : {metrics.box.map50:.4f}')
print(f'🏆 mAP@50-95 : {metrics.box.map:.4f}')